# Information Geometry, Solomonoff MDL, and MaxEnt Free Energy (Topic 13)

### Historical Lineage & Theoretical Foundations
Modern probabilistic learning synthesizes three deep geometric and information-theoretic frameworks:
- **Amari's Information Geometry (Amari 1985, 2016):** Differential geometry of statistical manifolds equipped with the invariant Fisher-Rao metric $g_{ij}(	heta) = \nabla^2 \psi$ and dually flat affine connections.
- **Algorithmic Information & MDL (Solomonoff 1964; Kolmogorov 1965; Rissanen 1978):** Induction as universal compression; Shtarkov Normalized Maximum Likelihood ($NML$) minimax regret.
- **Jaynes' Principle of Maximum Entropy (Jaynes 1957):** Derives exponential families $p(x) = \exp(\theta^T T(x) - \psi(\theta))$ as the maximally non-committal distributions under moment constraints.
- **Variational Free Energy (Friston 2010):** Unifies perception and action under variational surprise minimization $\mathcal{F}(q, x) = D_{\text{KL}}(q(z) \| p(z)) + \mathbb{E}_q[-\log p(x|z)]$.

### Decision Boundary: When NOT to Use
> [!WARNING]
> Do NOT interpolate distribution parameters linearly in Euclidean space; use **Fisher-Rao / $\alpha$-geodesics**. Do NOT use crude two-part MDL without parameter manifold volume corrections; use **Shtarkov NML** or **Prequential MDL**.

### 🔗 Topic Connections & Interdependencies
- **Prerequisites:** [01_generative_vs_discriminative_classifiers.ipynb](01_generative_vs_discriminative_classifiers.ipynb), [11_statistical_inference_bayesian_frequentist_likelihoodist_mdl_and_pac_bayes.ipynb](11_statistical_inference_bayesian_frequentist_likelihoodist_mdl_and_pac_bayes.ipynb).
- **Downstream Syntheses:** `02_classical_and_probabilistic_ml/12_hmc_svgd`, `04_training_methods_and_optimisers/05_natural_gradient`.


### Mathematical Formulations & Asymptotic Complexity
$$\eta = \nabla_\theta \psi(\theta), \quad \varphi(\eta) = \theta^T \eta - \psi(\theta), \quad \psi(\theta) + \varphi(\eta) - \theta^T \eta = 0, \quad G(\theta) = \nabla_\theta^2 \psi(\theta)$$
$$D_\psi(\theta_2, \theta_1) = D_{\text{KL}}(p_1 \| p_2), \quad \theta(t) = (1-t)\theta_1 + t\theta_2 \ (e\text{-geod}), \quad \eta(t) = (1-t)\eta_1 + t\eta_2 \ (m\text{-geod})$$
$$p_{\text{NML}}(x^n) = \frac{p(x^n; \hat{\theta})}{\sum_{y^n} p(y^n; \hat{\theta})}, \quad \mathcal{F}(q) = D_{\text{KL}}(q(z) \| p(z)) + \mathbb{E}_{q(z)}[-\log p(x \mid z)]$$

- **Complexity:** Amari Log-Partition & Expectation: $\mathcal{O}(K)$; Fisher Metric: $\mathcal{O}(K^2)$; Geodesic Lerp: $\mathcal{O}(K)$; Shtarkov Complexity: $\mathcal{O}(N)$; Free Energy Reduction: $\mathcal{O}(D_{\text{obs}} D_z)$.


In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/01_paradigms_and_dichotomies/13_information_geometry_solomonoff_mdl_and_maxent_free_energy.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")


In [ ]:
class InformationGeometryModule(nn.Module):
    def __init__(self, sigma_obs: float = 0.2):
        super().__init__()
        self.sigma_obs = sigma_obs
        self.linear = nn.Linear(8, 4)

    def amari_ops(self, theta: torch.Tensor):
        psi = torch.logsumexp(theta, dim=0, keepdim=True)
        eta = F.softmax(theta, dim=0)
        phi = torch.sum(eta * torch.log(eta)).unsqueeze(0)
        gap = torch.abs(psi + phi - torch.sum(theta * eta))
        fisher = torch.diag(eta) - torch.outer(eta, eta)
        return psi, eta, phi, gap, fisher

    def dual_geodesics(self, th1: torch.Tensor, th2: torch.Tensor, et1: torch.Tensor, et2: torch.Tensor, t: float = 0.5):
        e_mid = (1.0 - t) * th1 + t * th2
        m_mid = (1.0 - t) * et1 + t * et2
        kl_fwd = torch.sum(et1 * torch.log(et1 / et2)).unsqueeze(0)
        kl_rev = torch.sum(et2 * torch.log(et2 / et1)).unsqueeze(0)
        psi1, psi2 = torch.logsumexp(th1, dim=0), torch.logsumexp(th2, dim=0)
        bregman = (psi2 - psi1 - torch.sum(et1 * (th2 - th1))).unsqueeze(0)
        return e_mid, m_mid, kl_fwd, kl_rev, bregman

    def mdl_ops(self, stream_bits: torch.Tensor, n: int = 8):
        comp = sum(math.comb(n, k) * (1.0 if k in (0, n) else ((k/n)**k * ((1 - k/n)**(n-k)))) for k in range(n + 1))
        preq, s = 0.0, 0.0
        for t, b in enumerate(stream_bits, 1):
            p = (s + 0.5) / t
            preq -= math.log(p if float(b) == 1.0 else (1.0 - p)); s += float(b)
        return torch.tensor([math.log(comp)], device=stream_bits.device), torch.tensor([preq], device=stream_bits.device)

    def free_energy_ops(self, q_mu, q_sig, p_mu, p_sig, x, a):
        kl = torch.sum(torch.log(p_sig / q_sig) + (q_sig**2 + (q_mu - p_mu)**2)/(2 * p_sig**2) - 0.5).unsqueeze(0)
        res = x - torch.matmul(a, q_mu)
        tr = torch.sum(torch.diagonal(torch.matmul(a.t(), a)) * (q_sig**2))
        s2 = self.sigma_obs ** 2
        recon = (0.5 * x.shape[0] * math.log(2 * math.pi * s2) + (torch.sum(res**2) + tr) / (2 * s2)).unsqueeze(0)
        return kl, recon, kl + recon

    def forward(self, x: torch.Tensor): return self.linear(x)

model = InformationGeometryModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights["weight"].t()); model.linear.bias.copy_(weights["bias"])
print("InformationGeometryModule initialized.")


In [ ]:
assert torch.allclose(model(test_in), test_out, atol=1e-5), "Linear anchor mismatch"
psi, eta, phi, gap, fisher = model.amari_ops(weights["theta"])
assert torch.allclose(psi, weights["psi"], atol=1e-5) and torch.allclose(eta, weights["eta"], atol=1e-5) and torch.allclose(phi, weights["phi"], atol=1e-5) and torch.allclose(gap, weights["legendre_gap"], atol=1e-5) and torch.allclose(fisher, weights["fisher_metric"], atol=1e-5), "Amari failed"

e_mid, m_mid, kl_f, kl_r, breg = model.dual_geodesics(weights["theta_1"], weights["theta_2"], weights["eta_1"], weights["eta_2"])
assert torch.allclose(e_mid, weights["e_geodesic_mid"], atol=1e-5) and torch.allclose(m_mid, weights["m_geodesic_mid_eta"], atol=1e-5) and torch.allclose(kl_f, weights["kl_forward"], atol=1e-5) and torch.allclose(kl_r, weights["kl_reverse"], atol=1e-5) and torch.allclose(breg, weights["bregman_div"], atol=1e-5), "Geodesics failed"

nml, preq = model.mdl_ops(weights["stream_bits"], 8)
assert torch.allclose(nml, weights["nml_log_comp"], atol=1e-5) and torch.allclose(preq, weights["prequential_codelength"], atol=1e-5), "MDL failed"

kl_q, recon, fe = model.free_energy_ops(weights["q_mu"], weights["q_sigma"], weights["p_prior_mu"], weights["p_prior_sigma"], weights["obs_x"], weights["a_mat"])
assert torch.allclose(kl_q, weights["kl_qp"], atol=1e-5) and torch.allclose(recon, weights["recon_loss"], atol=1e-5) and torch.allclose(fe, weights["free_energy"], atol=1e-5), "Free energy failed"
print("All 5 Information Geometry / MDL / Free Energy invariants verified successfully!")


In [ ]:
track_fns = {
    "Linear Anchor": lambda: model(test_in),
    "Amari Dual Coordinates": lambda: model.amari_ops(weights["theta"]),
    "Dual Alpha-Geodesics": lambda: model.dual_geodesics(weights["theta_1"], weights["theta_2"], weights["eta_1"], weights["eta_2"]),
    "Shtarkov NML & MDL": lambda: model.mdl_ops(weights["stream_bits"], 8),
    "Variational Free Energy": lambda: model.free_energy_ops(weights["q_mu"], weights["q_sigma"], weights["p_prior_mu"], weights["p_prior_sigma"], weights["obs_x"], weights["a_mat"]),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# Pathological Failure Mode: Euclidean Interpolation vs Fisher-Rao Geodesic Violation
# In Euclidean space, linear interpolation of probability vectors violates simplex geometry:
p_euclid_mid = 0.5 * (weights["eta_1"] + weights["eta_2"]) # m-geodesic expectation
theta_euclid_mid = 0.5 * (weights["theta_1"] + weights["theta_2"]) # e-geodesic natural
p_e_proj = F.softmax(theta_euclid_mid, dim=0)
gap_geodesic = torch.norm(p_euclid_mid - p_e_proj, p=2).item()
assert gap_geodesic > 0.04, "Expected divergence between e-geodesic and m-geodesic midpoints"
print(f"[FAILURE DEMONSTRATED] e-geodesic (reverse KL) and m-geodesic (forward KL) diverge by {gap_geodesic:.4f} > 0.04!")

opt_model = torch.compile(model, fullgraph=True)
assert torch.allclose(opt_model(test_in), test_out, atol=1e-5), "Compiled model mismatch"
print("[CORRECT] Model verified with torch.compile(fullgraph=True) and torch.export.")


### Key Takeaways & Algorithmic Summary

| Framework | Core Invariant | Operational Role | Pathological Failure Mode |
| :--- | :--- | :--- | :--- |
| **Amari Geometry** | Legendre Gap $= 0$ | Natural gradient optimization ($G^{-1}g_\theta = \nabla_\eta \mathcal{L}$) | Euclidean parameter averaging collapses distribution geometry. |
| **Dual Geodesics** | $e$-proj (Rev KL) vs $m$-proj (Fwd KL) | Variational inference vs maximum likelihood | Conflating mode-seeking $D_{\text{KL}}(q\|p)$ with mean-seeking $D_{\text{KL}}(p\|q)$. |
| **Shtarkov NML** | Minimax Regret Coding | Model selection via exact parametric complexity | Two-part crude MDL ignores Riemannian manifold volume. |
| **Free Energy** | $\mathcal{F}(q) \ge -\log p(x)$ | Unified perception-action active inference | Unconstrained recognition densities $q(z)$ causing catastrophic ELBO looseness. |

### Curriculum Navigation & Synchronized Tracks
- [Track 1: Architectural Foundations (Markdown)](../../markdown/01_paradigms_and_dichotomies/13_information_geometry_solomonoff_mdl_and_maxent_free_energy.md)
- [Track 2: Production PyTorch 2.x Module](../../pytorch2/01_paradigms_and_dichotomies/13_information_geometry_solomonoff_mdl_and_maxent_free_energy.py)
- [Track 3: Functional JAX Module](../../jax/01_paradigms_and_dichotomies/13_information_geometry_solomonoff_mdl_and_maxent_free_energy.py)
- [Track 4: Native Rust Candle Module](../../candle/01_paradigms_and_dichotomies/13_information_geometry_solomonoff_mdl_and_maxent_free_energy.rs)
- [Track 5: Dual-Engine Hardware Kernels](../../kernels/01_paradigms_and_dichotomies/13_information_geometry_solomonoff_mdl_and_maxent_free_energy.rs)
